# Gold fact -- `dbo.fct_arrivals`

Badge reads at building entrances. Serves every occupancy, headcount and arrival-pattern measure on the executive dashboards.

**Grain:** One badge read. One row per cardholder, per building, per calendar date, per minute of day. NOT one row per person per day -- 83 of the 2,000 rows are a second entry by the same person to the same building on the same day, and they are real events, not duplicates. Any headcount must therefore be a DISTINCT COUNT of cardholders; counting rows over-reports occupancy by roughly 4%.

> GENERATED FILE -- DO NOT EDIT.
Produced by framework/generators/generate_notebooks.py from the project spec set. Edit the spec and regenerate; hand edits are overwritten and will fail the notebook-lint gate.


In [ ]:
# Parameters -- overridden per environment by the deployment pipeline.
# See 05-deployment.yaml `parameterisation`.
# Reads from lh_silver, writes to wh_gold. Both must be
# attached to this notebook; wh_gold must be the DEFAULT so an
# unqualified write cannot land in the wrong item.
target_item = "wh_gold"
source_item = "lh_silver"
environment = "dev"
dq_failure_action = "warn"

import sys
from datetime import datetime

from pyspark.sql import functions as F

from ttfabric.cleansing import RuleContext, get_rule
from ttfabric.quality import DQRunLog

load_id = f"load_{datetime.utcnow():%Y%m%d_%H%M%S}"

def resolve_table(name: str):
    """Resolve a spec table reference to a DataFrame.

    Deliberately UNQUALIFIED, so the read lands in the default lakehouse.

    Rules reference tables in their OWN layer -- enforce_referential_integrity
    against dim_products, recompute_total_from_lines against fct_order_items --
    and those peers live in the item this notebook writes to, not the one it
    reads its source from. Qualifying with source_item sent them to
    lh_bronze.dim_products, which does not and should not exist.

    The single cross-item read, this table's own bronze source, is qualified
    explicitly at the call site instead.
    """
    bare = name.split(".")[-1]
    return spark.read.table(bare)

ctx = RuleContext(
    spark=spark,
    load_id=load_id,
    environment=environment,
    table="fct_arrivals",
    resolve_table=resolve_table,
    apply_masking=(environment in ("uat", "prod")),
)

dq = DQRunLog(spark, load_id=load_id, layer="gold", table_name="fct_arrivals")
print(f"load_id={load_id}  environment={environment}  table=fct_arrivals")

from ttfabric.warehouse import gold_target

gold = gold_target(
    spark,
    warehouse="wh_gold",
    schema="dbo",
    write_mode="warehouse_connector",
)


In [ ]:
# ---- Read silver -------------------------------------------------
df = spark.read.table(f"{source_item}.stg_arrivals")


In [ ]:
# Business rule: arrival_flag
# 
df = df.withColumn("arrival_event", F.expr("""1"""))


In [ ]:
# Business rule: personnel_flag
# 
df = df.withColumn("personnel_arrival", F.expr("""CASE WHEN user_type = 1 THEN 1 ELSE 0 END"""))


In [ ]:
# Business rule: visitor_flag
# 
df = df.withColumn("visitor_arrival", F.expr("""CASE WHEN user_type = 2 THEN 1 ELSE 0 END"""))


In [ ]:
# Resolve card_holder_sk from dim_card_holder
from ttfabric.dimensions import lookup_surrogate_key

df = lookup_surrogate_key(
    df,
    dimension=gold.read("dim_card_holder"),
    surrogate_key="card_holder_sk",
    lookup_on="card_holder_guid",
    dimension_key="card_holder_guid",
    dimension_surrogate_key="card_holder_sk",
)


In [ ]:
# Resolve building_sk from dim_building
from ttfabric.dimensions import lookup_surrogate_key

df = lookup_surrogate_key(
    df,
    dimension=gold.read("dim_building"),
    surrogate_key="building_sk",
    lookup_on="building_key",
    dimension_key="building_key",
    dimension_surrogate_key="building_sk",
)


In [ ]:
# Resolve arrival_date_sk from dim_date
from ttfabric.dimensions import lookup_surrogate_key

df = lookup_surrogate_key(
    df,
    dimension=gold.read("dim_date"),
    surrogate_key="arrival_date_sk",
    lookup_on="date_key",
    dimension_key="date_key",
    dimension_surrogate_key="date_sk",
)


In [ ]:
# Resolve arrival_time_sk from dim_time
from ttfabric.dimensions import lookup_surrogate_key

df = lookup_surrogate_key(
    df,
    dimension=gold.read("dim_time"),
    surrogate_key="arrival_time_sk",
    lookup_on="time_key",
    dimension_key="time_key",
    dimension_surrogate_key="time_sk",
)


In [ ]:
# Resolve reader_sk from dim_reader
from ttfabric.dimensions import lookup_surrogate_key

df = lookup_surrogate_key(
    df,
    dimension=gold.read("dim_reader"),
    surrogate_key="reader_sk",
    lookup_on="reader_key",
    dimension_key="reader_key",
    dimension_surrogate_key="reader_sk",
)


In [ ]:
# ---- Rename to target names --------------------------------------
df = (df
    .withColumnRenamed("arrival_event", "arrival_count")
    .withColumnRenamed("personnel_arrival", "personnel_arrivals")
    .withColumnRenamed("visitor_arrival", "visitor_arrivals")
    .withColumnRenamed("used_presence_this_day", "presence_used_count")
    .withColumnRenamed("is_enrolled_today", "enrolled_today_count")
)


In [ ]:
# ---- Write -------------------------------------------------------
final_columns = ['arrival_id', 'user_type', 'user_type_label', 'hour_of_day', 'company_name', 'card_holder_sk', 'building_sk', 'arrival_date_sk', 'arrival_time_sk', 'reader_sk', 'arrival_count', 'personnel_arrivals', 'visitor_arrivals', 'presence_used_count', 'enrolled_today_count', 'arrival_event', 'personnel_arrival', 'visitor_arrival']
out = (df.select(*[c for c in final_columns if c in df.columns])
    .withColumn("_built_at", F.current_timestamp())
    .withColumn("_load_id", F.lit(load_id)))

gold.write(out, "fct_arrivals")
print(f"wrote {out.count():,} rows to fct_arrivals")


In [ ]:
# ---- Tests -------------------------------------------------------
#   GOLD-GRAIN-001: The declared grain holds -- one row per cardholder, building, date and minute. This is the test that keeps a distinct-count headcount honest.
#   GOLD-KEYS-001: Dimension keys resolve. A failed lookup yields the unknown member so the row survives, which means a fact with EVERY key unresolved passes a not-null check -- -1 is not null. This is what catches it.
#   GOLD-RECON-001: Personnel plus visitors equals all arrivals. F5's guidance is to reconcile on MONEY -- this dataset has no money column anywhere, so the additive measure is arrivals, and this is the check that stands in for it. A gap means user_type quarantining removed rows one side still counts.
#   GOLD-RECON-002: Gold ties back to ALL of silver, which is what makes a loss visible. Every silver arrival reaches the fact -- there is no join here that can discard one, and this asserts that stays true.
#   GOLD-DISTINCT-001: Distinct cardholders never exceeds arrivals, and is strictly less here because of the 83 re-entries. If these ever became equal, either the re-entries were silently deduplicated or the grain changed.
from ttfabric.quality import (assert_unique, assert_not_null,
                         assert_keys_resolve, assert_reconciles)

assert_unique(out, ['card_holder_sk', 'building_sk', 'arrival_date_sk', 'arrival_time_sk'])
assert_not_null(out, ['card_holder_sk', 'building_sk', 'arrival_date_sk', 'arrival_time_sk', 'reader_sk'])

# not_null is not enough: a failed lookup yields the unknown-member
# key, not a null, so a fact table with every key unresolved passes
# a not-null check while reporting everything against "Unknown".
#
# Optional keys are excluded: an event that has not happened has
# no date, and the unknown member is the correct destination.
assert_keys_resolve(out, ['card_holder_sk', 'building_sk', 'arrival_date_sk', 'arrival_time_sk', 'reader_sk'])

dq.record_input(out.count())
dq.record_output(out.count())
dq.flush()
